# Huấn luyện mô hình Titanic (Training Models)

## Import thư viện

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import joblib  # Để lưu model

## 1. Mục tiêu huấn luyện
+ **Mô tả**:
    + Chia train/val từ processed data.
    + Thử nhiều model: Logistic, RF, XGBoost, SVC, KNN.
    + Đánh giá bằng Accuracy, F1, ROC-AUC.
    + Chọn best model (dựa trên F1/ROC vì imbalance).
+ **Dữ liệu vào**: Từ processed.
+ **Kết quả**: Model tốt nhất lưu vào saved_models.

## 2. Load dữ liệu từ processed

In [12]:
# Load train processed (chỉ train có Survived)
train_path = '../data/processed/train_processed.csv'
df_train = pd.read_csv(train_path)

X = df_train.drop('Survived', axis=1)
y = df_train['Survived']

# Split train/val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 3. Huấn luyện và đánh giá models
+ Thử nhiều model và in metrics.

In [13]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'SVC': SVC(kernel='rbf', probability=True, random_state=42, C=1.0, gamma='scale'),
    'K-Neighbours': KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan', p=1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_prob) if y_prob is not None else None

    print(f"{name} Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {f'{roc_auc:.4f}' if roc_auc is not None else 'N/A'}")
    print("-" * 30)

    results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'ROC_AUC': roc_auc})

# Lưu metrics
pd.DataFrame(results).to_csv('../models/metrics/evaluation_results.csv', index=False)

Logistic Regression Results:
  Accuracy: 0.8101
  F1 Score: 0.7424
  ROC AUC:  0.8573
------------------------------
Random Forest Results:
  Accuracy: 0.8212
  F1 Score: 0.7612
  ROC AUC:  0.8387
------------------------------
XGBoost Results:
  Accuracy: 0.8324
  F1 Score: 0.7826
  ROC AUC:  0.8338
------------------------------
SVC Results:
  Accuracy: 0.8324
  F1 Score: 0.7761
  ROC AUC:  0.8431
------------------------------
K-Neighbours Results:
  Accuracy: 0.8268
  F1 Score: 0.7669
  ROC AUC:  0.8545
------------------------------


## 4. Chọn và lưu best model
+ Chọn SVC (dựa trên metrics tốt nhất).
+ Train full train data.

In [14]:
# Chọn best model (SVC ví dụ)
best_model = SVC(kernel='rbf', probability=True, random_state=42, C=1.0, gamma='scale')
best_model.fit(X, y)  # Train full train data

# Lưu model
joblib.dump(best_model, '../models/saved_models/best_svc_model.pkl')
print("Best model saved.")

Best model saved.


# Kết thúc

In [15]:
# Cách an toàn, đa nền tảng để xóa HTML hiện có và export notebook sang HTML
import os
import subprocess
from pathlib import Path
# Tính toán đường dẫn notebook và output tương đối với file notebook này
nb_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')
# Nếu chạy bên trong notebook, sử dụng thư mục làm việc hiện tại của server notebook
nb_dir = nb_dir if nb_dir.exists() else Path('.')
nb_path = nb_dir / 'train_model.ipynb'
out_path = nb_dir / 'train_model.html'
# Xóa file output hiện có nếu tồn tại
if out_path.exists():
    print(f'Removing existing file: {out_path}')
    out_path.unlink()
# Chạy nbconvert sử dụng subprocess để ổn định trên Windows
cmd = ['jupyter', 'nbconvert', str(nb_path), '--to', 'html']
print('Running:', ' '.join(cmd))
try:
    subprocess.run(cmd, check=True)
    print('Export complete:', out_path)
except subprocess.CalledProcessError as e:
    print('nbconvert failed with returncode', e.returncode)
    print('Ensure jupyter is available in the PATH of the environment running this notebook.')

Removing existing file: train_model.html
Running: jupyter nbconvert train_model.ipynb --to html
Export complete: train_model.html
